## **Import Library & Fungsi HOG**

In [1]:
import os
import cv2
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Fungsi Ekstraksi HOG (Sesuai PRD)
def extract_hog(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (128, 128)) # Wajib resize
    features = hog(
        img, 
        orientations=9, 
        pixels_per_cell=(8, 8), 
        cells_per_block=(2, 2), 
        block_norm="L2-Hys", 
        transform_sqrt=True, 
        feature_vector=True
    )
    return features

print("Semua library berhasil di-import!")

Semua library berhasil di-import!


## **Load Dataset & Ekstrak Fitur**

In [ ]:
DATASET_DIR = "../dataset" #dataset
CLASSES = ['Oval', 'Round', 'Square', 'Heart', 'Oblong']

X = []
y = []

print("Memproses dataset... (Mohon tunggu, ini mungkin butuh waktu)")
for label, class_name in enumerate(CLASSES):
    folder_path = os.path.join(DATASET_DIR, class_name)
    if not os.path.exists(folder_path):
        print(f"⚠️ Folder {class_name} tidak ditemukan, skip.")
        continue
        
    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)
        features = extract_hog(img_path)
        if features is not None:
            X.append(features)
            y.append(label)

X = np.array(X)
y = np.array(y)
print(f"✅ Selesai! Total gambar diproses: {len(X)}")

Memproses dataset... (Mohon tunggu, ini mungkin butuh waktu)
⚠️ Folder Oval tidak ditemukan, skip.
⚠️ Folder Round tidak ditemukan, skip.
⚠️ Folder Square tidak ditemukan, skip.
⚠️ Folder Heart tidak ditemukan, skip.
⚠️ Folder Oblong tidak ditemukan, skip.
✅ Selesai! Total gambar diproses: 0


## **Split Data & Training Model**

In [3]:
# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- TRAINING SVM ---
print("\n--- Training SVM ---")
svm_model = SVC(kernel='rbf', C=10, gamma='scale', probability=True)
svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)
print(f"Akurasi SVM: {accuracy_score(y_test, svm_pred) * 100:.2f}%")

# --- TRAINING RANDOM FOREST ---
print("\n--- Training Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
print(f"Akurasi RF: {accuracy_score(y_test, rf_pred) * 100:.2f}%")

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

## **Evaluasi Model**

Nampilin Classification Report & Gambar Confusion Matrix

In [4]:
# --- EVALUASI SVM ---
print("\n=== Classification Report SVM ===")
print(classification_report(y_test, svm_pred, target_names=CLASSES))

# --- EVALUASI RF ---
print("\n=== Classification Report RF ===")
print(classification_report(y_test, rf_pred, target_names=CLASSES))

# --- PLOT CONFUSION MATRIX ---
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
    plt.title(title)
    plt.ylabel('Actual Label')
    plt.xlabel('Predicted Label')
    plt.show()

print("\nConfusion Matrix SVM:")
plot_cm(y_test, svm_pred, "Confusion Matrix - SVM")

print("\nConfusion Matrix Random Forest:")
plot_cm(y_test, rf_pred, "Confusion Matrix - Random Forest")


=== Classification Report SVM ===


NameError: name 'y_test' is not defined

## **Simpan Model**

Simpan jadi file .joblib biar bisa dipake Backend

In [5]:
import os
os.makedirs('../saved_models', exist_ok=True)

# Simpan model SVM (karena biasanya SVM lebih bagus di HOG)
joblib.dump(svm_model, '../saved_models/svm_model.joblib')
joblib.dump(rf_model, '../saved_models/rf_model.joblib')

print("✅ Model SVM dan RF berhasil disimpan di folder saved_models/")

NameError: name 'svm_model' is not defined